# Free-GPU LLM API for agentic coding (Kaggle + Colab)

This notebook turns a Kaggle or Google Colab GPU session into a public, authenticated,
**OpenAI-compatible** endpoint (`/v1/chat/completions`, `/v1/completions`, `/v1/models`)
that coding agents such as Aider, OpenCode, Cline, Roo Code, Continue or the Codex CLI can use.
It detects the platform and GPU and picks a model that fits (`PRESET = "auto"`).

**Architecture**

```
agent(s) ──HTTPS──▶ tunnel (ngrok / Cloudflare) ──▶ :8000 async gateway (auth, least-busy routing, streaming)
                                                         ├──▶ :8001 vLLM (GPU 0 / or GPU 0+1 tensor-parallel)
                                                         └──▶ :8002 vLLM (GPU 1)            [replica mode]
                                         or a single llama.cpp server on :8001 (layers split across GPUs)
```

**How it handles many connections**

* **vLLM continuous batching + PagedAttention**: dozens of requests share each forward pass instead of queueing.
* **Prefix caching**: agents resend the same long system prompt / repo context every turn; the shared prefix is computed once.
* **Chunked prefill**: a huge prompt from one client doesn't freeze token streaming for everyone else.
* **Replica mode**: with two GPUs and a model that fits on one, two independent engines double throughput; the gateway sends each request to the least-busy one and retries on another if one is down.
* **Async gateway**: one aiohttp process with unlimited keep-alive connections, true streaming passthrough (SSE), and API-key auth.

**Secrets** (all optional; Kaggle: *Add-ons → Secrets*; Colab: the 🔑 icon in the left sidebar, then allow notebook access)

* `API_KEY`: the key clients must send. If missing, a random one is generated and printed.
* `NGROK_TOKEN`: ngrok authtoken (recommended tunnel; supports streaming). `NGROK_DOMAIN`: your free static domain.
* `CF_TUNNEL_TOKEN` + `CF_HOSTNAME`: a named Cloudflare tunnel whose public hostname points to `http://localhost:8000`.
* `HF_TOKEN`: only needed for gated models.

### On Kaggle

1. *Settings → Accelerator*: **GPU T4 ×2**. (P100 only works with the llama.cpp backend: vLLM needs a T4 or newer.)
2. *Settings → Internet*: **On** (requires a phone-verified Kaggle account).
3. Run all cells. For unattended use, choose **Save Version → Save & Run All (Commit)**: it then runs up to 12 h without a browser tab open.

Kaggle gives ~30 GPU-hours/week and a 12 h session limit.

### On Colab

1. *Runtime → Change runtime type*: **T4 GPU** (free), or **L4** / **A100** (paid plans).
2. *Runtime → Run all*. Keep the tab open: Colab disconnects idle sessions, and there is no background "commit" run like Kaggle's
   (some paid plans offer background execution).

| Colab GPU | Auto preset | What to expect |
|---|---|---|
| T4 (free, 16 GB) | `qwen2.5-coder-7b` | One engine; a handful of agents at once |
| L4 (24 GB) | `qwen3-coder-30b` on vLLM, 32k context | Best agent model; a few agents at once |
| A100 (40/80 GB) | `qwen3-coder-30b` on vLLM, 64k context | Many agents at once |

> **Colab terms:** Colab's FAQ restricts some activities on the free tier, including running web services unrelated to interactive
> notebook use and connecting through remote proxies. Serving an API through a tunnel may fall under that, and Colab can end the
> runtime or limit the account. Paid plans have fewer restrictions. Check the current FAQ before relying on this.
> On either platform, keep the API key secret: anyone with the URL and key can use your GPU.

## 1. Configuration

`PRESET = "auto"` picks a model for your GPU (shown in the next cell's output). To choose yourself, set a preset name or `"custom"`.
On T4/P100 everything runs in fp16, because those GPUs have no bf16 support.

| Preset | Weights | Engine | Notes |
|---|---|---|---|
| `qwen3-coder-30b`   | ~17-19 GB | vLLM, falling back to llama.cpp | **Best for agents.** MoE with 3B active parameters, so it's fast. Trained for tool use. Auto on L4/A100. Needs ~20 GB. |
| `qwen2.5-coder-14b` | ~10 GB | vLLM, split across GPUs | Auto on Kaggle T4 ×2. Proven on T4. |
| `qwen2.5-coder-7b`  | ~5.5 GB | vLLM, one replica per GPU | Auto on a single T4 (Colab free). Most simultaneous users. |
| `qwen2.5-coder-32b` | ~19 GB | vLLM, split across GPUs | Strong coder; short context, few users. |
| `qwen3-8b`          | ~6 GB | vLLM, one replica per GPU | General + coding, hybrid thinking. |

**Engines.** `backend="vllm"` gives the highest throughput, but vLLM needs a T4 or newer.
`backend="llamacpp"` builds llama.cpp from source (~6-10 min the first time) and serves a GGUF file. It runs on T4 and P100 and is very reliable,
but has lower total throughput under heavy load. `backend="auto"` tries vLLM and falls back to llama.cpp if vLLM fails to start.
On a P100, vLLM is skipped automatically.

In [ ]:
# ============================ USER SETTINGS ============================
PRESET = "auto"                   # "auto" picks by GPU; or one of PRESETS below, or "custom"

# Only used when PRESET == "custom"
CUSTOM = dict(
    backend="vllm",               # vllm | llamacpp | auto
    model="Qwen/Qwen2.5-Coder-14B-Instruct-AWQ",   # Hugging Face repo for vLLM
    gguf_repo=None, gguf_quant="Q4_K_M",           # Hugging Face GGUF repo + quant for llama.cpp
    tool_parser="hermes",         # vLLM tool parser: hermes | qwen3_coder | llama3_json | mistral | ...
    max_model_len=32768,
    mode="tp",                    # vLLM: "tp" = one engine across all GPUs, "replicas" = one engine per GPU
    extra_args=[],
)

SERVED_MODEL_NAME = "cloud-coder"    # name clients put in "model"
MAX_NUM_SEQS      = 48               # vLLM: max concurrent sequences per engine (lower if OOM)
GPU_MEM_UTIL      = 0.92             # vLLM: fraction of each GPU it may use
VLLM_VERSION      = ""               # "" = latest; pin e.g. "0.10.1.1" if a new release breaks on T4
EXTRA_VLLM_ARGS   = []               # e.g. ["--enforce-eager"] to save VRAM (slower)

LLAMACPP_PARALLEL     = 4            # llama.cpp: simultaneous requests (slots); more slots = more memory
LLAMACPP_CTX_PER_SLOT = 32768        # llama.cpp: context length each slot gets
LLAMACPP_KV_TYPE      = "q8_0"       # llama.cpp: KV-cache precision (q8_0 halves memory vs f16)

TUNNEL            = "auto"           # auto | ngrok | cloudflare | cloudflare-quick | none
GATEWAY_PORT      = 8000
# =======================================================================

PRESETS = {
    "qwen3-coder-30b":   dict(backend="auto",
                              model="cpatonn/Qwen3-Coder-30B-A3B-Instruct-AWQ-4bit",
                              gguf_repo="unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF", gguf_quant="Q4_K_M",
                              tool_parser="qwen3_coder", max_model_len=65536, mode="tp",
                              extra_args=["--max-num-seqs", "16"]),
    "qwen2.5-coder-14b": dict(backend="vllm", model="Qwen/Qwen2.5-Coder-14B-Instruct-AWQ",
                              gguf_repo="Qwen/Qwen2.5-Coder-14B-Instruct-GGUF", gguf_quant="Q4_K_M",
                              tool_parser="hermes", max_model_len=32768, mode="tp", extra_args=[]),
    "qwen2.5-coder-7b":  dict(backend="vllm", model="Qwen/Qwen2.5-Coder-7B-Instruct-AWQ",
                              gguf_repo="Qwen/Qwen2.5-Coder-7B-Instruct-GGUF", gguf_quant="Q4_K_M",
                              tool_parser="hermes", max_model_len=32768, mode="replicas", extra_args=[]),
    "qwen2.5-coder-32b": dict(backend="vllm", model="Qwen/Qwen2.5-Coder-32B-Instruct-AWQ",
                              tool_parser="hermes", max_model_len=16384, mode="tp", extra_args=[]),
    "qwen3-8b":          dict(backend="vllm", model="Qwen/Qwen3-8B-AWQ", tool_parser="hermes",
                              max_model_len=32768, mode="replicas",
                              extra_args=["--reasoning-parser", "qwen3"]),
}

## 2. Platform, GPU and secrets

In [ ]:
import os, sys, secrets, subprocess
import torch

IS_KAGGLE = os.path.isdir("/kaggle/working")
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
PLATFORM = "kaggle" if IS_KAGGLE else "colab" if IS_COLAB else "other"
WORK_DIR = "/kaggle/working" if IS_KAGGLE else "/content" if IS_COLAB else os.getcwd()

def get_secret(name):
    if IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        except Exception:
            pass
    if IS_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:   # secret missing, or notebook access not granted
            pass
    return os.environ.get(name)

API_KEY         = get_secret("API_KEY") or ("sk-" + secrets.token_urlsafe(24))
NGROK_TOKEN     = get_secret("NGROK_TOKEN")
CF_TUNNEL_TOKEN = get_secret("CF_TUNNEL_TOKEN")
HF_TOKEN        = get_secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,compute_cap",
                          "--format=csv"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    pass
NUM_GPUS = torch.cuda.device_count()
assert NUM_GPUS > 0, ("No GPU found. Kaggle: Settings → Accelerator → GPU T4 x2. "
                      "Colab: Runtime → Change runtime type → T4/L4/A100 GPU.")
GPU_CC   = torch.cuda.get_device_capability(0)
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB  = min(torch.cuda.get_device_properties(i).total_memory for i in range(NUM_GPUS)) / 1024**3
TOTAL_VRAM_GB = VRAM_GB * NUM_GPUS
print(f"Platform: {PLATFORM} | {NUM_GPUS}× {GPU_NAME} ({VRAM_GB:.0f} GB, sm_{GPU_CC[0]}{GPU_CC[1]})")

# ---- choose the preset ----
if PRESET == "auto":
    if GPU_CC >= (8, 0) and TOTAL_VRAM_GB >= 22:
        PRESET = "qwen3-coder-30b"        # L4 / A100 / H100: vLLM's fast 4-bit MoE kernels work here
    elif NUM_GPUS >= 2 and VRAM_GB >= 14:
        PRESET = "qwen2.5-coder-14b"      # Kaggle T4 x2
    else:
        PRESET = "qwen2.5-coder-7b"       # single T4 (Colab free), P100 (llama.cpp)
CFG = dict(CUSTOM if PRESET == "custom" else PRESETS[PRESET])
CFG.setdefault("gguf_repo", None); CFG.setdefault("gguf_quant", "Q4_K_M")

# ---- fit the preset to the hardware ----
if PRESET == "qwen3-coder-30b":
    if TOTAL_VRAM_GB < 22:
        raise SystemExit("qwen3-coder-30b needs ~20 GB of GPU memory (T4 x2, L4 or A100). Use a smaller preset.")
    if TOTAL_VRAM_GB >= 38:                # A100 40GB reports ~39.4 GiB
        CFG["extra_args"] = []            # plenty of KV cache: keep MAX_NUM_SEQS
    elif NUM_GPUS == 1:
        CFG["max_model_len"] = 32768      # L4: ~4 GB left for KV cache after the weights
if PRESET == "qwen2.5-coder-14b" and TOTAL_VRAM_GB < 20:
    CFG["max_model_len"] = 12288          # single 16 GB GPU: little room left for KV cache
VLLM_DTYPE = "auto" if GPU_CC >= (8, 0) else "half"   # T4/P100 have no bf16

print("Preset:", PRESET)
print(CFG)
print("Secrets found:", {k: bool(v) for k, v in
      dict(API_KEY=get_secret("API_KEY"), NGROK_TOKEN=NGROK_TOKEN,
           CF_TUNNEL_TOKEN=CF_TUNNEL_TOKEN, HF_TOKEN=HF_TOKEN).items()})

## 3. Install dependencies (~3-5 min)

vLLM is installed unless the backend is `llamacpp`. llama.cpp is only built if it's needed (see section 5).

In [ ]:
BACKEND = CFG["backend"]
if GPU_CC < (7, 0) and BACKEND != "llamacpp":
    if not CFG["gguf_repo"]:
        raise SystemExit(f"GPU compute capability {GPU_CC} is too old for vLLM and this preset has no GGUF. "
                         "Switch to a T4 or newer, or pick a preset with a llama.cpp fallback.")
    print(f"GPU compute capability {GPU_CC} < 7.0: vLLM unsupported, using llama.cpp")
    BACKEND = "llamacpp"
if BACKEND == "auto" and not CFG["gguf_repo"]:
    BACKEND = "vllm"
print("Backend:", BACKEND)

spec = f"vllm=={VLLM_VERSION}" if VLLM_VERSION else "vllm"
pkgs = "aiohttp pyngrok openai huggingface_hub" + ("" if BACKEND == "llamacpp" else f' "{spec}"')
!pip install -q -U {pkgs} 2>&1 | grep -vE "dependency resolver|incompatible|^\s*$" | tail -n 5
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
!cloudflared --version
if BACKEND != "llamacpp":
    !python -c "import vllm; print('vLLM', vllm.__version__)"

LLAMA_DIR = "/root/llama.cpp"
LLAMA_SERVER = f"{LLAMA_DIR}/build/bin/llama-server"

def ensure_llamacpp():
    """Build llama-server with CUDA for this GPU's architecture only (keeps the build to a few minutes)."""
    if os.path.exists(LLAMA_SERVER):
        return
    arch = f"{GPU_CC[0]}{GPU_CC[1]}"
    nvcc = "/usr/local/cuda/bin/nvcc"
    assert os.path.exists(nvcc), "CUDA toolkit (nvcc) not found; cannot build llama.cpp"
    print(f"Building llama.cpp for sm_{arch} (one-time, ~6-10 min)...")
    cmds = [
        f"rm -rf {LLAMA_DIR} && git clone -q --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA_DIR}",
        f"cd {LLAMA_DIR} && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={arch} "
        f"-DCMAKE_CUDA_COMPILER={nvcc} -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /root/llama_cmake.log 2>&1",
        f"cd {LLAMA_DIR} && cmake --build build --target llama-server -j$(nproc) > /root/llama_build.log 2>&1",
    ]
    for c in cmds:
        if subprocess.run(c, shell=True).returncode != 0:
            print(subprocess.run("tail -n 40 /root/llama_cmake.log /root/llama_build.log", shell=True,
                                 capture_output=True, text=True).stdout)
            raise RuntimeError("llama.cpp build failed: " + c.split("&&")[-1][:80])
    print("Built", LLAMA_SERVER)

if BACKEND == "llamacpp":
    ensure_llamacpp()

## 4. Gateway (auth + least-busy load balancing + streaming)

A small aiohttp reverse proxy. It
* checks `Authorization: Bearer <key>` (or `x-api-key`),
* sends each request to the backend with the fewest in-flight requests,
* retries on another backend if one refuses the connection,
* streams responses chunk by chunk, so SSE token streaming works,
* serves an unauthenticated `GET /health` for monitoring.

In [ ]:
GATEWAY_SRC = r"""
import asyncio, os, sys, time
from aiohttp import web, ClientSession, ClientTimeout, TCPConnector, ClientConnectorError

API_KEY  = os.environ["GW_API_KEY"]
BACKENDS = os.environ["GW_BACKENDS"].split(",")
PORT     = int(os.environ.get("GW_PORT", "8000"))
inflight = {b: 0 for b in BACKENDS}
stats    = {"total": 0, "errors": 0, "started": time.time()}
HOP = {"connection", "keep-alive", "proxy-authenticate", "proxy-authorization", "te",
       "trailers", "transfer-encoding", "upgrade", "host", "content-length",
       "authorization", "x-api-key"}

def authorized(req):
    auth = req.headers.get("Authorization", "")
    key = auth[7:] if auth.lower().startswith("bearer ") else req.headers.get("x-api-key", "")
    return key == API_KEY

async def health(req):
    return web.json_response({"ok": True, "inflight": inflight, **stats})

async def proxy(req):
    if req.method == "OPTIONS":
        return web.Response(headers={"Access-Control-Allow-Origin": "*",
            "Access-Control-Allow-Headers": "*", "Access-Control-Allow-Methods": "*"})
    if not authorized(req):
        return web.json_response({"error": {"message": "Invalid API key", "type": "auth_error"}}, status=401)
    stats["total"] += 1
    body = await req.read()
    headers = {k: v for k, v in req.headers.items() if k.lower() not in HOP}
    tried = set()
    while True:
        candidates = [b for b in BACKENDS if b not in tried]
        if not candidates:
            stats["errors"] += 1
            return web.json_response({"error": {"message": "No backend available"}}, status=503)
        backend = min(candidates, key=lambda b: inflight[b])
        tried.add(backend)
        inflight[backend] += 1
        resp = None
        try:
            async with req.app["session"].request(req.method, backend + req.rel_url.path_qs,
                                                  data=body, headers=headers) as up:
                resp = web.StreamResponse(status=up.status, headers={
                    k: v for k, v in up.headers.items() if k.lower() not in HOP})
                resp.headers["Access-Control-Allow-Origin"] = "*"
                resp.headers["X-Accel-Buffering"] = "no"
                await resp.prepare(req)
                async for chunk in up.content.iter_any():
                    await resp.write(chunk)
                await resp.write_eof()
                return resp
        except ClientConnectorError:
            if resp is None:
                continue          # backend down before anything was sent: try the next one
            raise
        except (ConnectionResetError, asyncio.CancelledError):
            raise                 # client went away
        finally:
            inflight[backend] -= 1

async def on_startup(app):
    app["session"] = ClientSession(connector=TCPConnector(limit=0, keepalive_timeout=60),
                                   timeout=ClientTimeout(total=None, sock_connect=5, sock_read=1800),
                                   auto_decompress=False)

async def on_cleanup(app):
    await app["session"].close()

app = web.Application(client_max_size=256 * 1024 ** 2)
app.on_startup.append(on_startup)
app.on_cleanup.append(on_cleanup)
app.router.add_get("/health", health)
app.router.add_route("*", "/{tail:.*}", proxy)
web.run_app(app, host="0.0.0.0", port=PORT, backlog=4096, access_log=None)
"""
with open(f"{WORK_DIR}/gateway.py", "w") as f:
    f.write(GATEWAY_SRC)
GATEWAY_PATH = os.path.abspath(f.name)
print("Wrote", GATEWAY_PATH)

## 5. Launch the model server(s)

* **vLLM, `tp` mode**: one engine split across all GPUs (bigger models, longer context).
* **vLLM, `replicas` mode**: one engine per GPU (more total throughput when the model fits on one GPU).
* **llama.cpp**: one server with the model's layers split across all GPUs, serving `LLAMACPP_PARALLEL` requests at once.

With `backend="auto"`, if vLLM fails to start, its log is printed, its weights are deleted to free disk,
and llama.cpp takes over. The first launch downloads the weights, so it can take 5-15 minutes.

In [ ]:
import time, requests, shutil, glob

LOG_DIR = f"{WORK_DIR}/logs"
os.makedirs(LOG_DIR, exist_ok=True)

def pkill(*patterns):
    for p in patterns:
        subprocess.run(["pkill", "-f", p])

def log_path(e):
    return f"{LOG_DIR}/{e['kind']}_{e['port']}.log"

# ---------- vLLM ----------
def vllm_engines():
    mode = CFG["mode"] if NUM_GPUS > 1 else "tp"
    if mode == "tp":
        return [dict(kind="vllm", gpus=",".join(map(str, range(NUM_GPUS))), tp=NUM_GPUS, port=8001)]
    return [dict(kind="vllm", gpus=str(i), tp=1, port=8001 + i) for i in range(NUM_GPUS)]

def vllm_cmd(e):
    return [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", CFG["model"],
        "--served-model-name", SERVED_MODEL_NAME, CFG["model"],
        "--host", "127.0.0.1", "--port", str(e["port"]),
        "--tensor-parallel-size", str(e["tp"]),
        "--dtype", VLLM_DTYPE,                   # "half" on T4/P100 (no bf16)
        "--max-model-len", str(CFG["max_model_len"]),
        "--max-num-seqs", str(MAX_NUM_SEQS),
        "--gpu-memory-utilization", str(GPU_MEM_UTIL),
        "--enable-prefix-caching",               # agents resend the same long prefix every turn
        "--enable-chunked-prefill",              # long prompts don't stall other streams
        "--enable-auto-tool-choice", "--tool-call-parser", CFG["tool_parser"],
        "--disable-log-requests",
        "--trust-remote-code",
        *CFG.get("extra_args", []), *EXTRA_VLLM_ARGS,   # later flags override earlier ones
    ]

# ---------- llama.cpp ----------
GGUF_PATH = None

def download_gguf():
    global GGUF_PATH
    if GGUF_PATH:
        return GGUF_PATH
    from huggingface_hub import snapshot_download
    q = CFG["gguf_quant"]
    print(f"Downloading {CFG['gguf_repo']} ({q})...")
    d = snapshot_download(CFG["gguf_repo"], allow_patterns=[f"*{q}*.gguf", f"*{q.lower()}*.gguf"],
                          local_dir=f"/root/models/{CFG['gguf_repo'].replace('/', '__')}")
    files = sorted(glob.glob(f"{d}/**/*.gguf", recursive=True))
    files = [f for f in files if "mmproj" not in f]
    assert files, f"No *{q}*.gguf file in {CFG['gguf_repo']}; check gguf_quant"
    GGUF_PATH = files[0]          # split files: llama.cpp loads the rest from the first shard
    print("GGUF:", GGUF_PATH)
    return GGUF_PATH

def llamacpp_cmd(e):
    ctx = min(CFG["max_model_len"], LLAMACPP_CTX_PER_SLOT)
    return [
        LLAMA_SERVER, "-m", GGUF_PATH,
        "--host", "127.0.0.1", "--port", str(e["port"]),
        "--alias", SERVED_MODEL_NAME,
        "-ngl", "999",                           # all layers on GPU, split across GPUs by layer
        "-c", str(ctx * LLAMACPP_PARALLEL), "-np", str(LLAMACPP_PARALLEL),   # total ctx is shared by the slots
        "--flash-attn", "on",
        "--cache-type-k", LLAMACPP_KV_TYPE, "--cache-type-v", LLAMACPP_KV_TYPE,
        "--jinja",                               # model's own chat template, needed for tool calls
        "--metrics",
    ]

# ---------- common ----------
def start_engine(e):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=e["gpus"], VLLM_WORKER_MULTIPROC_METHOD="spawn")
    cmd = vllm_cmd(e) if e["kind"] == "vllm" else llamacpp_cmd(e)
    e["proc"] = subprocess.Popen(cmd, env=env, stdout=open(log_path(e), "a"),
                                 stderr=subprocess.STDOUT, start_new_session=True)
    print(f"Started {e['kind']} on GPU(s) {e['gpus']} → :{e['port']} (pid {e['proc'].pid})")

def wait_ready(e, timeout=2400):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if e["proc"].poll() is not None:
            print(open(log_path(e)).read()[-4000:])
            raise RuntimeError(f"{e['kind']} :{e['port']} exited; see log above")
        try:
            if requests.get(f"http://127.0.0.1:{e['port']}/health", timeout=2).ok:
                print(f"{e['kind']} :{e['port']} ready after {time.time()-t0:.0f}s")
                return
        except requests.RequestException:
            pass
        time.sleep(5)
    raise TimeoutError(f"{e['kind']} :{e['port']} not ready after {timeout}s")

def stop_engines(engines):
    for e in engines:
        if e.get("proc") and e["proc"].poll() is None:
            os.killpg(e["proc"].pid, 9)
    time.sleep(5)

def launch_vllm():
    engines = vllm_engines()
    print("vLLM:", " ".join(vllm_cmd(engines[0])))
    for i, e in enumerate(engines):
        start_engine(e)
        if i == 0 and len(engines) > 1:
            wait_ready(e)   # first engine downloads the weights; the others reuse the HF cache
    for e in engines:
        wait_ready(e)
    return engines

def launch_llamacpp():
    ensure_llamacpp()
    download_gguf()
    e = dict(kind="llamacpp", gpus=",".join(map(str, range(NUM_GPUS))), port=8001)
    print("llama.cpp:", " ".join(llamacpp_cmd(e)))
    start_engine(e)
    wait_ready(e)
    return [e]

pkill("vllm.entrypoints.openai.api_server", "llama-server", "gateway.py"); time.sleep(3)
print("Backend:", BACKEND, "| model:", CFG["model"] if BACKEND != "llamacpp" else CFG["gguf_repo"])

if BACKEND == "llamacpp":
    engines = launch_llamacpp()
else:
    try:
        engines = launch_vllm()
    except Exception as err:
        if BACKEND != "auto":
            raise
        print(f"\n*** vLLM failed ({err}). Falling back to llama.cpp ***\n")
        pkill("vllm.entrypoints.openai.api_server"); time.sleep(5)
        cache = os.path.expanduser("~/.cache/huggingface/hub/models--" + CFG["model"].replace("/", "--"))
        shutil.rmtree(cache, ignore_errors=True)   # free disk for the GGUF
        BACKEND = "llamacpp"
        engines = launch_llamacpp()

SERVED_CTX = (CFG["max_model_len"] if BACKEND != "llamacpp"
              else min(CFG["max_model_len"], LLAMACPP_CTX_PER_SLOT))
print(f"\nServing with {BACKEND}; context per request: {SERVED_CTX}")
!tail -n 5 {LOG_DIR}/*_8001.log

## 6. Start the gateway

In [ ]:
gw_env = dict(os.environ, GW_API_KEY=API_KEY, GW_PORT=str(GATEWAY_PORT),
              GW_BACKENDS=",".join(f"http://127.0.0.1:{e['port']}" for e in engines))
gw_log = open(f"{LOG_DIR}/gateway.log", "a")
gateway_proc = subprocess.Popen(["python", GATEWAY_PATH], env=gw_env, stdout=gw_log,
                                stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(30):
    try:
        if requests.get(f"http://127.0.0.1:{GATEWAY_PORT}/health", timeout=2).ok:
            break
    except requests.RequestException:
        time.sleep(1)
print(requests.get(f"http://127.0.0.1:{GATEWAY_PORT}/health").json())
r = requests.get(f"http://127.0.0.1:{GATEWAY_PORT}/v1/models", headers={"Authorization": f"Bearer {API_KEY}"})
print(r.status_code, [m["id"] for m in r.json()["data"]])

## 7. Expose it publicly

`TUNNEL="auto"` picks the first available of:

1. **ngrok** (`NGROK_TOKEN` secret). Free account; supports streaming; you can use your free static domain.
2. **Cloudflare named tunnel** (`CF_TUNNEL_TOKEN` secret). Stable hostname on your own domain; best for many users.
3. **Cloudflare quick tunnel**. No account needed, random `*.trycloudflare.com` URL. Quick tunnels are meant for testing:
   Cloudflare limits concurrent requests and may not stream Server-Sent Events, so tokens can arrive all at once.

In [ ]:
import re
PUBLIC_URL = None
tunnel_proc = None

def start_ngrok():
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    ngrok.kill()
    domain = get_secret("NGROK_DOMAIN")          # optional: your free static domain
    t = ngrok.connect(GATEWAY_PORT, "http", **({"domain": domain} if domain else {}))
    return t.public_url

def start_cloudflare_named():
    global tunnel_proc
    tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "run",
                                    "--token", CF_TUNNEL_TOKEN],
                                   stdout=open(f"{LOG_DIR}/cloudflared.log", "a"),
                                   stderr=subprocess.STDOUT, start_new_session=True)
    time.sleep(8)
    return get_secret("CF_HOSTNAME") or "(your Cloudflare tunnel hostname)"

def start_cloudflare_quick():
    global tunnel_proc
    logf = f"{LOG_DIR}/cloudflared_quick.log"
    open(logf, "w").close()
    tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate",
                                    "--url", f"http://127.0.0.1:{GATEWAY_PORT}"],
                                   stdout=open(logf, "a"), stderr=subprocess.STDOUT,
                                   start_new_session=True)
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(logf).read())
        if m:
            return m.group(0)
        time.sleep(1)
    raise RuntimeError("Quick tunnel URL not found; see " + logf)

pkill("cloudflared")
choice = TUNNEL
if choice == "auto":
    choice = "ngrok" if NGROK_TOKEN else "cloudflare" if CF_TUNNEL_TOKEN else "cloudflare-quick"
if choice == "ngrok":
    PUBLIC_URL = start_ngrok()
elif choice == "cloudflare":
    PUBLIC_URL = start_cloudflare_named()
elif choice == "cloudflare-quick":
    PUBLIC_URL = start_cloudflare_quick()
else:
    PUBLIC_URL = f"http://127.0.0.1:{GATEWAY_PORT}"
if not PUBLIC_URL.startswith("http"):
    PUBLIC_URL = "https://" + PUBLIC_URL
BASE_URL = PUBLIC_URL.rstrip("/") + "/v1"

print(f"""
Tunnel:   {choice}
Base URL: {BASE_URL}
API key:  {API_KEY}
Model:    {SERVED_MODEL_NAME}
""")

## 8. Smoke tests: chat, tool calling, concurrency

In [ ]:
from openai import OpenAI
time.sleep(3)
client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=600)

r = client.chat.completions.create(model=SERVED_MODEL_NAME, max_tokens=200, messages=[
    {"role": "user", "content": "Write a Python one-liner that reverses the words in a string."}])
print(r.choices[0].message.content)

tools = [{"type": "function", "function": {
    "name": "read_file", "description": "Read a file from the repository",
    "parameters": {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]}}}]
r = client.chat.completions.create(model=SERVED_MODEL_NAME, tools=tools, tool_choice="auto", messages=[
    {"role": "user", "content": "Open src/main.py so you can look at it."}])
print("tool calls:", r.choices[0].message.tool_calls)

print("streaming: ", end="")
for chunk in client.chat.completions.create(model=SERVED_MODEL_NAME, stream=True, max_tokens=40,
        messages=[{"role": "user", "content": "Count from 1 to 15."}]):
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

In [ ]:
# Concurrency benchmark: N simultaneous agents through the public URL
import asyncio
from openai import AsyncOpenAI

N_CLIENTS, MAX_TOKENS = 32, 256

async def one(aclient, i):
    t0 = time.time()
    r = await aclient.chat.completions.create(model=SERVED_MODEL_NAME, max_tokens=MAX_TOKENS,
        messages=[{"role": "user", "content": f"Task #{i}: write a Python function implementing binary search, with docstring."}])
    return r.usage.completion_tokens, time.time() - t0

async def bench():
    aclient = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=900)
    t0 = time.time()
    res = await asyncio.gather(*(one(aclient, i) for i in range(N_CLIENTS)), return_exceptions=True)
    wall = time.time() - t0
    ok = [x for x in res if not isinstance(x, Exception)]
    toks = sum(t for t, _ in ok)
    print(f"{len(ok)}/{N_CLIENTS} succeeded in {wall:.1f}s | {toks/wall:.0f} tok/s aggregate | "
          f"avg latency {sum(l for _, l in ok)/max(len(ok),1):.1f}s")
    for x in res:
        if isinstance(x, Exception):
            print("error:", repr(x)[:200]); break

await bench()

## 9. Connect your coding agent

Every tool below speaks the OpenAI API. Use the **Base URL**, **API key** and **model** printed above.
Run the next cell for copy-paste snippets with your values filled in.

In [ ]:
print(f"""# ---- Generic (most tools read these) ----
export OPENAI_API_BASE="{BASE_URL}"
export OPENAI_BASE_URL="{BASE_URL}"
export OPENAI_API_KEY="{API_KEY}"

# ---- Aider ----
aider --model openai/{SERVED_MODEL_NAME}

# ---- OpenCode: ~/.config/opencode/opencode.json ----
{{
  "$schema": "https://opencode.ai/config.json",
  "provider": {{
    "cloudgpu": {{
      "npm": "@ai-sdk/openai-compatible",
      "name": "Cloud GPU",
      "options": {{ "baseURL": "{BASE_URL}", "apiKey": "{API_KEY}" }},
      "models": {{ "{SERVED_MODEL_NAME}": {{ "name": "{CFG['model']}" }} }}
    }}
  }}
}}

# ---- Codex CLI: ~/.codex/config.toml ----
model = "{SERVED_MODEL_NAME}"
model_provider = "cloudgpu"
[model_providers.cloudgpu]
name = "Cloud GPU"
base_url = "{BASE_URL}"
env_key = "OPENAI_API_KEY"
wire_api = "chat"

# ---- Cline / Roo Code / Continue / Kilo ----
Provider: "OpenAI Compatible" | Base URL: {BASE_URL} | API key: {API_KEY} | Model: {SERVED_MODEL_NAME}
Context window: {SERVED_CTX}
""")

## 10. Keep alive + monitor (+ auto-restart crashed engines)

This cell blocks. It prints running/waiting requests and KV-cache usage every minute and restarts any engine that dies.
Interrupt it to get the notebook back; the servers keep running in the background.

In [ ]:
def metrics(port):
    try:
        txt = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=3).text
    except requests.RequestException:
        return "down"
    def g(name):
        vals = [float(l.split()[-1]) for l in txt.splitlines()
                if l.startswith(name) and not l.startswith("#")]
        return sum(vals) if vals else float("nan")
    def first(*names):
        for n in names:
            v = g(n)
            if v == v:          # not NaN
                return v
        return float("nan")
    run  = first("vllm:num_requests_running", "llamacpp:requests_processing")
    wait = first("vllm:num_requests_waiting", "llamacpp:requests_deferred")
    kv   = first("vllm:gpu_cache_usage_perc", "vllm:kv_cache_usage_perc", "llamacpp:kv_cache_usage_ratio")
    return f"run={run:.0f} wait={wait:.0f} kv={kv*100:.0f}%"

print(f"Serving {BASE_URL}  (model '{SERVED_MODEL_NAME}')")
try:
    while True:
        for e in engines:
            if e["proc"].poll() is not None:
                print(time.strftime("%H:%M:%S"), f"engine :{e['port']} died, restarting")
                start_engine(e); wait_ready(e)
        if gateway_proc.poll() is not None:
            print("gateway died, restarting")
            gateway_proc = subprocess.Popen(["python", GATEWAY_PATH], env=gw_env, stdout=gw_log,
                                            stderr=subprocess.STDOUT, start_new_session=True)
        if tunnel_proc is not None and tunnel_proc.poll() is not None:
            print("tunnel died; re-run section 7 (a quick tunnel gets a NEW URL)")
        gw = requests.get(f"http://127.0.0.1:{GATEWAY_PORT}/health", timeout=3).json()
        print(time.strftime("%H:%M:%S"), f"total={gw['total']} errors={gw['errors']} |",
              " | ".join(f":{e['port']} {metrics(e['port'])}" for e in engines), flush=True)
        time.sleep(60)
except KeyboardInterrupt:
    print("Monitor stopped (servers still running).")

## Troubleshooting

* **CUDA out of memory at startup**: lower `MAX_NUM_SEQS` (e.g. 16), `max_model_len`, or `GPU_MEM_UTIL`, or add `--enforce-eager` to `EXTRA_VLLM_ARGS`.
* **Attention-backend / "compute capability" errors on T4**: pin an older vLLM with `VLLM_VERSION = "0.10.1.1"`, or set `os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"` (or `"FLEX_ATTENTION"`) before section 5.
* **Tool calls come back as plain text**: the `tool_parser` must match the model's chat template (`hermes` for Qwen2.5/Qwen3, `qwen3_coder` for Qwen3-Coder, `llama3_json` for Llama 3.x, `mistral` for Mistral).
* **Requests queue (`wait` > 0 for long)**: the GPU is saturated. Use a smaller model in `replicas` mode, or lower `max_tokens` on the client.
* **`qwen3-coder-30b` on vLLM fails on T4**: expected on some vLLM versions (4-bit MoE kernels mostly target newer GPUs). With `backend="auto"` it falls back to llama.cpp by itself; set `backend="llamacpp"` in the preset to skip the vLLM attempt next time.
* **llama.cpp out of memory**: lower `LLAMACPP_PARALLEL` or `LLAMACPP_CTX_PER_SLOT`, or use a smaller quant (`gguf_quant="Q3_K_M"`).
* **llama.cpp rejects a flag** (e.g. `--flash-attn on`): flags change between releases. Check `llamacpp_8001.log` and edit `llamacpp_cmd`.
* **llama.cpp build fails**: the logs are `/root/llama_cmake.log` and `/root/llama_build.log`.
* **Responses don't stream through the quick tunnel**: use ngrok or a named Cloudflare tunnel.
* **Colab disconnected**: Colab stops idle sessions and the URL dies with it. Re-run all cells; a quick tunnel gets a new URL, ngrok with `NGROK_DOMAIN` keeps the same one.
* **Colab secret not found**: open the 🔑 sidebar and turn on "Notebook access" for each secret.
* **Logs**: `/kaggle/working/logs/` on Kaggle, `/content/logs/` on Colab (`vllm_8001.log` or `llamacpp_8001.log`, `gateway.log`, `cloudflared*.log`).

In [ ]:
# Shut everything down (frees the GPU without restarting the kernel)
pkill("vllm.entrypoints.openai.api_server", "llama-server", "gateway.py", "cloudflared", "ngrok")